In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Gráfica comparativa por estación.

Para cada estación:
1. Se selecciona una semana completa a partir de WEEK_START.
2. Se representa la serie observada de O3 durante esa semana.
3. Se superponen, en el mismo gráfico, las predicciones de RF, XGBoost, GRU y LSTM
   únicamente para los últimos 3 días de esa semana.

Notas:
- RF y XGBoost usan la ventana de 72 horas sin normalización.
- GRU y LSTM usan scaler_X para la entrada y scaler_y para desescalar la salida.
- Si falta alguno de los modelos, la estación se omite para mantener la comparación completa.
"""

import os
import pickle
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model

# =============================================================================
# CONFIGURACIÓN
# =============================================================================

STATIONS = [
    "T1_E1_Alicante",
    "T1_E2_Elda",
    "T3_E1_Valencia",
    "T3_E2_Buñol",
    "T5_E1_Castellon",
    "T6_E2_Coratxa",
]

BASE_DIR = os.path.expanduser("/Volumes/copia_seguridad1/enviar_benja/carpeta sin título/clean/Finales/")

ENCODED_DIR = os.path.join(BASE_DIR, "encoded", "ml", "global")
OUTPUT_DIR = os.path.join(BASE_DIR, "comparative_forecasts_final")

TREE_MODEL_BASE = {
    "RF": os.path.join(BASE_DIR, "models", "random_forest", "global"),
    "XGBoost": os.path.join(BASE_DIR, "models", "xgboost", "global"),
}

RNN_MODEL_BASE = {
    "GRU": os.path.join(BASE_DIR, "models", "rnn", "global"),
    "LSTM": os.path.join(BASE_DIR, "models", "lstm", "global"),
}

SCALERS_BASE = os.path.join(BASE_DIR, "windows_partitioned", "global", "dl")

WINDOW_IN = 72
WINDOW_OUT = 72
WEEK_LENGTH_HOURS = 168
FORECAST_HOURS = 72

# Cambia esta fecha por la semana que quieras analizar
WEEK_START = pd.Timestamp("2024-05-01 00:00:00")

MODEL_STYLE = {
    "RF": {
        "color": "tab:red",
        "linestyle": "--",
        "linewidth": 2.0,
        "label": "RF",
    },
    "XGBoost": {
        "color": "tab:green",
        "linestyle": "-.",
        "linewidth": 2.0,
        "label": "XGBoost",
    },
    "GRU": {
        "color": "tab:orange",
        "linestyle": ":",
        "linewidth": 2.5,
        "label": "GRU",
    },
    "LSTM": {
        "color": "tab:purple",
        "linestyle": (0, (5, 2)),
        "linewidth": 2.0,
        "label": "LSTM",
    },
}

# =============================================================================
# FUNCIONES
# =============================================================================

def load_original_data(station: str) -> pd.DataFrame:
    """
    Carga el CSV de una estación, detecta la columna temporal y deja el índice
    en frecuencia horaria. Convierte los datos a numéricos e interpola huecos.
    """
    csv_path = os.path.join(ENCODED_DIR, f"{station}.csv")
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"No se encontró el archivo: {csv_path}")

    df = pd.read_csv(csv_path)

    time_col = None
    for col in df.columns:
        if col.lower() in {"timestamp", "datetime", "fecha", "time"}:
            time_col = col
            break
    if time_col is None:
        time_col = df.columns[0]

    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
    df = df.dropna(subset=[time_col]).copy()
    df = df.set_index(time_col).sort_index()

    df = df.asfreq("h")
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.interpolate(method="time", limit_direction="both")
    df = df.bfill().ffill()

    return df


def get_week_slice(df: pd.DataFrame, week_start: pd.Timestamp) -> pd.DataFrame:
    """
    Devuelve exactamente 168 horas desde week_start.
    """
    week_end = week_start + pd.Timedelta(hours=WEEK_LENGTH_HOURS - 1)
    week_df = df.loc[week_start:week_end].copy()

    if len(week_df) != WEEK_LENGTH_HOURS:
        raise ValueError(
            f"La semana seleccionada no contiene {WEEK_LENGTH_HOURS} horas exactas. "
            f"Se han obtenido {len(week_df)} filas."
        )

    return week_df


def get_input_window(df: pd.DataFrame, forecast_start: pd.Timestamp) -> pd.DataFrame:
    """
    Extrae las 72 horas previas al inicio del pronóstico.
    """
    start = forecast_start - pd.Timedelta(hours=WINDOW_IN)
    end = forecast_start - pd.Timedelta(hours=1)
    window_df = df.loc[start:end].copy()

    if len(window_df) != WINDOW_IN:
        raise ValueError(
            f"La ventana de entrada no contiene {WINDOW_IN} horas exactas. "
            f"Se han obtenido {len(window_df)} filas."
        )

    return window_df


def load_tree_model(model_kind: str, station: str):
    """
    Carga un modelo de tipo árbol serializado con pickle.
    """
    model_path = os.path.join(TREE_MODEL_BASE[model_kind], station, "model.pkl")
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"No se encontró el modelo: {model_path}")

    with open(model_path, "rb") as f:
        model = pickle.load(f)

    return model


def load_rnn_bundle(model_kind: str, station: str):
    """
    Carga un modelo Keras y sus scalers asociados.
    """
    model_path = os.path.join(RNN_MODEL_BASE[model_kind], station, "model.keras")
    scaler_X_path = os.path.join(SCALERS_BASE, station, "scaler_X.pkl")
    scaler_y_path = os.path.join(SCALERS_BASE, station, "scaler_y.pkl")

    if not os.path.exists(model_path):
        raise FileNotFoundError(f"No se encontró el modelo: {model_path}")
    if not os.path.exists(scaler_X_path):
        raise FileNotFoundError(f"No se encontró scaler_X: {scaler_X_path}")
    if not os.path.exists(scaler_y_path):
        raise FileNotFoundError(f"No se encontró scaler_y: {scaler_y_path}")

    model = load_model(model_path)

    with open(scaler_X_path, "rb") as f:
        scaler_X = pickle.load(f)

    with open(scaler_y_path, "rb") as f:
        scaler_y = pickle.load(f)

    return model, scaler_X, scaler_y


def predict_tree_model(model, input_window: pd.DataFrame) -> pd.Series:
    """
    Predicción para RF y XGBoost.
    """
    flat_input = input_window.values.reshape(1, -1)
    pred = model.predict(flat_input)

    if pred.ndim == 2:
        pred = pred.flatten()

    return pd.Series(pred[:WINDOW_OUT])


def predict_rnn_model(model, input_window: pd.DataFrame, scaler_X, scaler_y) -> pd.Series:
    """
    Predicción para GRU y LSTM.
    """
    X = scaler_X.transform(input_window.values)
    X = X.reshape(1, input_window.shape[0], input_window.shape[1])

    pred_norm = model.predict(X, verbose=0)

    if pred_norm.ndim == 2:
        pred_norm = pred_norm.flatten()

    pred_original = scaler_y.inverse_transform(pred_norm.reshape(-1, 1)).flatten()
    return pd.Series(pred_original[:WINDOW_OUT])


def plot_station_comparison(
    station: str,
    week_o3: pd.Series,
    predictions: dict,
    week_start: pd.Timestamp,
):
    """
    Genera una sola figura con la semana observada y las predicciones de los 4 modelos
    para los últimos 3 días.
    """
    forecast_start = week_start + pd.Timedelta(hours=WEEK_LENGTH_HOURS - FORECAST_HOURS)
    forecast_index = pd.date_range(start=forecast_start, periods=FORECAST_HOURS, freq="h")
    forecast_end = forecast_index[-1]

    fig, ax = plt.subplots(figsize=(15, 6))

    ax.plot(
        week_o3.index,
        week_o3.values,
        color="black",
        linewidth=1.8,
        label="O3 observado",
    )

    ax.axvspan(
        forecast_start,
        forecast_end,
        alpha=0.08,
        color="gray",
        label="Periodo de predicción",
    )

    for model_name in ["RF", "XGBoost", "GRU", "LSTM"]:
        pred = predictions[model_name]
        style = MODEL_STYLE[model_name]

        ax.plot(
            forecast_index,
            pred.values,
            color=style["color"],
            linestyle=style["linestyle"],
            linewidth=style["linewidth"],
            label=style["label"],
        )

    ax.axvline(
        forecast_start,
        color="dimgray",
        linestyle=":",
        linewidth=1.5,
    )

    ax.set_title(
        f"Serie semanal de O3 y predicciones de modelos\n"
        f"{station}  |  Semana: {week_start.strftime('%Y-%m-%d')}"
    )
    ax.set_xlabel("Fecha y hora")
    ax.set_ylabel("Concentración de O3")
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=3)
    fig.tight_layout()

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    out_path = os.path.join(
        OUTPUT_DIR,
        f"{station}_week_{week_start.strftime('%Y%m%d')}_comparison.png",
    )
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    print(f"Gráfica guardada: {out_path}")


# =============================================================================
# PROGRAMA PRINCIPAL
# =============================================================================

def main():
    print("Generando gráficas comparativas por estación...")
    print(f"Directorio de salida: {OUTPUT_DIR}")
    print(f"Semana seleccionada: {WEEK_START}")

    for station in STATIONS:
        print(f"\nProcesando {station}")

        try:
            df = load_original_data(station)
        except Exception as e:
            print(f"  Error al cargar datos: {e}")
            continue

        if "O3" not in df.columns:
            print("  La columna O3 no existe en el CSV. Se omite la estación.")
            continue

        try:
            week_df = get_week_slice(df, WEEK_START)
        except Exception as e:
            print(f"  Error al extraer la semana: {e}")
            continue

        forecast_start = WEEK_START + pd.Timedelta(hours=WEEK_LENGTH_HOURS - FORECAST_HOURS)

        try:
            input_window = get_input_window(df, forecast_start)
        except Exception as e:
            print(f"  Error al extraer la ventana de entrada: {e}")
            continue

        predictions = {}

        try:
            rf_model = load_tree_model("RF", station)
            predictions["RF"] = predict_tree_model(rf_model, input_window)
        except Exception as e:
            print(f"  RF no disponible: {e}")

        try:
            xgb_model = load_tree_model("XGBoost", station)
            predictions["XGBoost"] = predict_tree_model(xgb_model, input_window)
        except Exception as e:
            print(f"  XGBoost no disponible: {e}")

        try:
            gru_model, gru_scaler_X, gru_scaler_y = load_rnn_bundle("GRU", station)
            predictions["GRU"] = predict_rnn_model(gru_model, input_window, gru_scaler_X, gru_scaler_y)
        except Exception as e:
            print(f"  GRU no disponible: {e}")

        try:
            lstm_model, lstm_scaler_X, lstm_scaler_y = load_rnn_bundle("LSTM", station)
            predictions["LSTM"] = predict_rnn_model(lstm_model, input_window, lstm_scaler_X, lstm_scaler_y)
        except Exception as e:
            print(f"  LSTM no disponible: {e}")

        if len(predictions) != 4:
            print("  No están disponibles los cuatro modelos. Se omite la estación.")
            continue

        try:
            plot_station_comparison(
                station=station,
                week_o3=week_df["O3"],
                predictions=predictions,
                week_start=WEEK_START,
            )
        except Exception as e:
            print(f"  Error al generar la gráfica: {e}")

    print("\nProceso completado.")


if __name__ == "__main__":
    main()

Generando gráficas comparativas por estación...
Directorio de salida: /Volumes/copia_seguridad1/enviar_benja/carpeta sin título/clean/Finales/comparative_forecasts_2025
Semana seleccionada: 2025-05-01 00:00:00

Procesando T1_E1_Alicante
Gráfica guardada: /Volumes/copia_seguridad1/enviar_benja/carpeta sin título/clean/Finales/comparative_forecasts_2025/T1_E1_Alicante_week_20250501_comparison.png

Procesando T1_E2_Elda
Gráfica guardada: /Volumes/copia_seguridad1/enviar_benja/carpeta sin título/clean/Finales/comparative_forecasts_2025/T1_E2_Elda_week_20250501_comparison.png

Procesando T3_E1_Valencia
Gráfica guardada: /Volumes/copia_seguridad1/enviar_benja/carpeta sin título/clean/Finales/comparative_forecasts_2025/T3_E1_Valencia_week_20250501_comparison.png

Procesando T3_E2_Buñol
Gráfica guardada: /Volumes/copia_seguridad1/enviar_benja/carpeta sin título/clean/Finales/comparative_forecasts_2025/T3_E2_Buñol_week_20250501_comparison.png

Procesando T5_E1_Castellon
Gráfica guardada: /Volum

In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Gráfica comparativa por estación para N semanas consecutivas.
Compatible con Jupyter Notebook y línea de comandos.
"""

import os
import pickle
import argparse
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

# Reducir la verbosidad de TensorFlow
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')

from tensorflow.keras.models import load_model

# =============================================================================
# CONFIGURACIÓN
# =============================================================================

STATIONS = [
    "T1_E1_Alicante",
    "T1_E2_Elda",
    "T3_E1_Valencia",
    "T3_E2_Buñol",
    "T5_E1_Castellon",
    "T6_E2_Coratxa",
]

BASE_DIR = os.path.expanduser("/Volumes/copia_seguridad1/enviar_benja/carpeta sin título/clean/Finales/")

ENCODED_DIR = os.path.join(BASE_DIR, "encoded", "ml", "global")
OUTPUT_DIR = os.path.join(BASE_DIR, "comparative_forecasts_multiple_weeks")

TREE_MODEL_BASE = {
    "RF": os.path.join(BASE_DIR, "models", "random_forest", "global"),
    "XGBoost": os.path.join(BASE_DIR, "models", "xgboost", "global"),
}

RNN_MODEL_BASE = {
    "GRU": os.path.join(BASE_DIR, "models", "rnn", "global"),
    "LSTM": os.path.join(BASE_DIR, "models", "lstm", "global"),
}

SCALERS_BASE = os.path.join(BASE_DIR, "windows_partitioned", "global", "dl")

WINDOW_IN = 72
WINDOW_OUT = 72
WEEK_LENGTH_HOURS = 168
FORECAST_HOURS = 72

# Cambia esta fecha por la primera semana que quieras analizar
WEEK_START = pd.Timestamp("2024-05-01 00:00:00")

# Valor por defecto si no se especifica argumento
DEFAULT_NUM_WEEKS = 8

MODEL_STYLE = {
    "RF": {
        "color": "tab:red",
        "linestyle": "--",
        "linewidth": 2.0,
        "label": "RF",
    },
    "XGBoost": {
        "color": "tab:green",
        "linestyle": "-.",
        "linewidth": 2.0,
        "label": "XGBoost",
    },
    "GRU": {
        "color": "tab:orange",
        "linestyle": ":",
        "linewidth": 2.5,
        "label": "GRU",
    },
    "LSTM": {
        "color": "tab:purple",
        "linestyle": (0, (5, 2)),
        "linewidth": 2.0,
        "label": "LSTM",
    },
}

# =============================================================================
# FUNCIONES
# =============================================================================

def parse_arguments():
    """Lee argumentos de línea de comandos ignorando los desconocidos (compatible con Jupyter)."""
    parser = argparse.ArgumentParser(
        description="Genera gráficas comparativas de pronóstico de O3 para múltiples semanas."
    )
    parser.add_argument(
        "-n", "--num-weeks",
        type=int,
        default=DEFAULT_NUM_WEEKS,
        help=f"Número de semanas a procesar (por defecto: {DEFAULT_NUM_WEEKS})"
    )
    # parse_known_args ignora argumentos extra como los que añade Jupyter
    args, _ = parser.parse_known_args()
    return args


def load_original_data(station: str) -> pd.DataFrame:
    """
    Carga el CSV de una estación, detecta la columna temporal y deja el índice
    en frecuencia horaria. Convierte los datos a numéricos e interpola huecos.
    """
    csv_path = os.path.join(ENCODED_DIR, f"{station}.csv")
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"No se encontró el archivo: {csv_path}")

    df = pd.read_csv(csv_path)

    time_col = None
    for col in df.columns:
        if col.lower() in {"timestamp", "datetime", "fecha", "time"}:
            time_col = col
            break
    if time_col is None:
        time_col = df.columns[0]

    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
    df = df.dropna(subset=[time_col]).copy()
    df = df.set_index(time_col).sort_index()

    df = df.asfreq("h")
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.interpolate(method="time", limit_direction="both")
    df = df.bfill().ffill()

    return df


def get_week_slice(df: pd.DataFrame, week_start: pd.Timestamp) -> pd.DataFrame:
    """
    Devuelve exactamente 168 horas desde week_start.
    """
    week_end = week_start + pd.Timedelta(hours=WEEK_LENGTH_HOURS - 1)
    week_df = df.loc[week_start:week_end].copy()

    if len(week_df) != WEEK_LENGTH_HOURS:
        raise ValueError(
            f"La semana seleccionada no contiene {WEEK_LENGTH_HOURS} horas exactas. "
            f"Se han obtenido {len(week_df)} filas."
        )

    return week_df


def get_input_window(df: pd.DataFrame, forecast_start: pd.Timestamp) -> pd.DataFrame:
    """
    Extrae las 72 horas previas al inicio del pronóstico.
    """
    start = forecast_start - pd.Timedelta(hours=WINDOW_IN)
    end = forecast_start - pd.Timedelta(hours=1)
    window_df = df.loc[start:end].copy()

    if len(window_df) != WINDOW_IN:
        raise ValueError(
            f"La ventana de entrada no contiene {WINDOW_IN} horas exactas. "
            f"Se han obtenido {len(window_df)} filas."
        )

    return window_df


def load_tree_model(model_kind: str, station: str):
    """
    Carga un modelo de tipo árbol serializado con pickle.
    """
    model_path = os.path.join(TREE_MODEL_BASE[model_kind], station, "model.pkl")
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"No se encontró el modelo: {model_path}")

    with open(model_path, "rb") as f:
        model = pickle.load(f)

    return model


def load_rnn_bundle(model_kind: str, station: str):
    """
    Carga un modelo Keras y sus scalers asociados.
    """
    model_path = os.path.join(RNN_MODEL_BASE[model_kind], station, "model.keras")
    scaler_X_path = os.path.join(SCALERS_BASE, station, "scaler_X.pkl")
    scaler_y_path = os.path.join(SCALERS_BASE, station, "scaler_y.pkl")

    if not os.path.exists(model_path):
        raise FileNotFoundError(f"No se encontró el modelo: {model_path}")
    if not os.path.exists(scaler_X_path):
        raise FileNotFoundError(f"No se encontró scaler_X: {scaler_X_path}")
    if not os.path.exists(scaler_y_path):
        raise FileNotFoundError(f"No se encontró scaler_y: {scaler_y_path}")

    model = load_model(model_path)

    with open(scaler_X_path, "rb") as f:
        scaler_X = pickle.load(f)

    with open(scaler_y_path, "rb") as f:
        scaler_y = pickle.load(f)

    return model, scaler_X, scaler_y


def predict_tree_model(model, input_window: pd.DataFrame) -> pd.Series:
    """
    Predicción para RF y XGBoost.
    """
    flat_input = input_window.values.reshape(1, -1)
    pred = model.predict(flat_input)

    if pred.ndim == 2:
        pred = pred.flatten()

    return pd.Series(pred[:WINDOW_OUT])


def predict_rnn_model(model, input_window: pd.DataFrame, scaler_X, scaler_y) -> pd.Series:
    """
    Predicción para GRU y LSTM.
    """
    X = scaler_X.transform(input_window.values)
    X = X.reshape(1, input_window.shape[0], input_window.shape[1])

    pred_norm = model.predict(X, verbose=0)

    if pred_norm.ndim == 2:
        pred_norm = pred_norm.flatten()

    pred_original = scaler_y.inverse_transform(pred_norm.reshape(-1, 1)).flatten()
    return pd.Series(pred_original[:WINDOW_OUT])


def plot_station_comparison(
    station: str,
    week_o3: pd.Series,
    predictions: dict,
    week_start: pd.Timestamp,
):
    """
    Genera una sola figura con la semana observada y las predicciones de los 4 modelos
    para los últimos 3 días.
    """
    forecast_start = week_start + pd.Timedelta(hours=WEEK_LENGTH_HOURS - FORECAST_HOURS)
    forecast_index = pd.date_range(start=forecast_start, periods=FORECAST_HOURS, freq="h")
    forecast_end = forecast_index[-1]

    fig, ax = plt.subplots(figsize=(15, 6))

    ax.plot(
        week_o3.index,
        week_o3.values,
        color="black",
        linewidth=1.8,
        label="O3 observado",
    )

    ax.axvspan(
        forecast_start,
        forecast_end,
        alpha=0.08,
        color="gray",
        label="Periodo de predicción",
    )

    for model_name in ["RF", "XGBoost", "GRU", "LSTM"]:
        pred = predictions[model_name]
        style = MODEL_STYLE[model_name]

        ax.plot(
            forecast_index,
            pred.values,
            color=style["color"],
            linestyle=style["linestyle"],
            linewidth=style["linewidth"],
            label=style["label"],
        )

    ax.axvline(
        forecast_start,
        color="dimgray",
        linestyle=":",
        linewidth=1.5,
    )

    ax.set_title(
        f"Serie semanal de O3 y predicciones de modelos\n"
        f"{station}  |  Semana: {week_start.strftime('%Y-%m-%d')}"
    )
    ax.set_xlabel("Fecha y hora")
    ax.set_ylabel("Concentración de O3")
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=3)
    fig.tight_layout()

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    out_path = os.path.join(
        OUTPUT_DIR,
        f"{station}_week_{week_start.strftime('%Y%m%d')}_comparison.png",
    )
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    print(f"Gráfica guardada: {out_path}")


# =============================================================================
# PROGRAMA PRINCIPAL
# =============================================================================

def main():
    args = parse_arguments()
    NUM_WEEKS = args.num_weeks

    print(f"Generando gráficas comparativas por estación para {NUM_WEEKS} semanas consecutivas...")
    print(f"Directorio de salida: {OUTPUT_DIR}")
    print(f"Primera semana: {WEEK_START}")

    # Generar las fechas de inicio de las N semanas
    week_starts = [WEEK_START + pd.Timedelta(days=7*i) for i in range(NUM_WEEKS)]

    for station in STATIONS:
        print(f"\n=== Procesando estación: {station} ===")

        # Cargar datos originales una sola vez por estación
        try:
            df = load_original_data(station)
        except Exception as e:
            print(f"  Error al cargar datos de {station}: {e}")
            continue

        if "O3" not in df.columns:
            print("  La columna O3 no existe en el CSV. Se omite la estación.")
            continue

        # Cargar los cuatro modelos (solo una vez por estación)
        models = {}
        try:
            models["RF"] = load_tree_model("RF", station)
        except Exception as e:
            print(f"  RF no disponible: {e}")
        try:
            models["XGBoost"] = load_tree_model("XGBoost", station)
        except Exception as e:
            print(f"  XGBoost no disponible: {e}")
        try:
            gru_model, gru_sx, gru_sy = load_rnn_bundle("GRU", station)
            models["GRU"] = (gru_model, gru_sx, gru_sy)
        except Exception as e:
            print(f"  GRU no disponible: {e}")
        try:
            lstm_model, lstm_sx, lstm_sy = load_rnn_bundle("LSTM", station)
            models["LSTM"] = (lstm_model, lstm_sx, lstm_sy)
        except Exception as e:
            print(f"  LSTM no disponible: {e}")

        # Verificar que los cuatro modelos estén cargados
        if len(models) != 4:
            print("  No están disponibles los cuatro modelos. Se omite la estación.")
            continue

        # Procesar cada semana
        for week_start in week_starts:
            print(f"  Procesando semana: {week_start.date()}")

            try:
                week_df = get_week_slice(df, week_start)
            except Exception as e:
                print(f"    Error al extraer la semana: {e}")
                continue

            forecast_start = week_start + pd.Timedelta(hours=WEEK_LENGTH_HOURS - FORECAST_HOURS)

            try:
                input_window = get_input_window(df, forecast_start)
            except Exception as e:
                print(f"    Error al extraer la ventana de entrada: {e}")
                continue

            predictions = {}

            # Predicción RF
            try:
                predictions["RF"] = predict_tree_model(models["RF"], input_window)
            except Exception as e:
                print(f"    Error en predicción RF: {e}")

            # Predicción XGBoost
            try:
                predictions["XGBoost"] = predict_tree_model(models["XGBoost"], input_window)
            except Exception as e:
                print(f"    Error en predicción XGBoost: {e}")

            # Predicción GRU
            try:
                gru_model, gru_sx, gru_sy = models["GRU"]
                predictions["GRU"] = predict_rnn_model(gru_model, input_window, gru_sx, gru_sy)
            except Exception as e:
                print(f"    Error en predicción GRU: {e}")

            # Predicción LSTM
            try:
                lstm_model, lstm_sx, lstm_sy = models["LSTM"]
                predictions["LSTM"] = predict_rnn_model(lstm_model, input_window, lstm_sx, lstm_sy)
            except Exception as e:
                print(f"    Error en predicción LSTM: {e}")

            if len(predictions) != 4:
                print("    No se obtuvieron las cuatro predicciones. Se omite esta semana.")
                continue

            try:
                plot_station_comparison(
                    station=station,
                    week_o3=week_df["O3"],
                    predictions=predictions,
                    week_start=week_start,
                )
            except Exception as e:
                print(f"    Error al generar la gráfica: {e}")

    print("\nProceso completado.")


if __name__ == "__main__":
    main()

Generando gráficas comparativas por estación para 8 semanas consecutivas...
Directorio de salida: /Volumes/copia_seguridad1/enviar_benja/carpeta sin título/clean/Finales/comparative_forecasts_multiple_weeks
Primera semana: 2024-05-01 00:00:00

=== Procesando estación: T1_E1_Alicante ===
  Procesando semana: 2024-05-01
Gráfica guardada: /Volumes/copia_seguridad1/enviar_benja/carpeta sin título/clean/Finales/comparative_forecasts_multiple_weeks/T1_E1_Alicante_week_20240501_comparison.png
  Procesando semana: 2024-05-08
Gráfica guardada: /Volumes/copia_seguridad1/enviar_benja/carpeta sin título/clean/Finales/comparative_forecasts_multiple_weeks/T1_E1_Alicante_week_20240508_comparison.png
  Procesando semana: 2024-05-15
Gráfica guardada: /Volumes/copia_seguridad1/enviar_benja/carpeta sin título/clean/Finales/comparative_forecasts_multiple_weeks/T1_E1_Alicante_week_20240515_comparison.png
  Procesando semana: 2024-05-22
Gráfica guardada: /Volumes/copia_seguridad1/enviar_benja/carpeta sin tí

In [6]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Script de diagnóstico para comparar predicciones de RF, XGBoost, GRU y LSTM
con valores reales de O3. Compatible con MinMaxScaler.
"""

import os
import pickle
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import mean_squared_error, mean_absolute_error

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')

# =============================================================================
# CONFIGURACIÓN - MODIFICAR AQUÍ LOS PARÁMETROS
# =============================================================================
STATION = "T1_E1_Alicante"   # Cambiar por la estación deseada
WEEK_START_STR = "2024-05-01 00:00:00"  # Cambiar por la fecha deseada

BASE_DIR = os.path.expanduser("/Volumes/copia_seguridad1/enviar_benja/carpeta sin título/clean/Finales/")
ENCODED_DIR = os.path.join(BASE_DIR, "encoded", "ml", "global")
TREE_MODEL_BASE = {
    "RF": os.path.join(BASE_DIR, "models", "random_forest", "global"),
    "XGBoost": os.path.join(BASE_DIR, "models", "xgboost", "global"),
}
RNN_MODEL_BASE = {
    "GRU": os.path.join(BASE_DIR, "models", "rnn", "global"),
    "LSTM": os.path.join(BASE_DIR, "models", "lstm", "global"),
}
SCALERS_BASE = os.path.join(BASE_DIR, "windows_partitioned", "global", "dl")

WINDOW_IN = 72
WINDOW_OUT = 72
WEEK_LENGTH_HOURS = 168
FORECAST_HOURS = 72

# =============================================================================
# FUNCIONES DE CARGA
# =============================================================================
def load_original_data(station: str) -> pd.DataFrame:
    csv_path = os.path.join(ENCODED_DIR, f"{station}.csv")
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"No se encontró el archivo: {csv_path}")
    df = pd.read_csv(csv_path)
    time_col = None
    for col in df.columns:
        if col.lower() in {"timestamp", "datetime", "fecha", "time"}:
            time_col = col
            break
    if time_col is None:
        time_col = df.columns[0]
    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
    df = df.dropna(subset=[time_col]).copy()
    df = df.set_index(time_col).sort_index()
    df = df.asfreq("h")
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.interpolate(method="time", limit_direction="both")
    df = df.bfill().ffill()
    return df

def get_week_slice(df: pd.DataFrame, week_start: pd.Timestamp) -> pd.DataFrame:
    week_end = week_start + pd.Timedelta(hours=WEEK_LENGTH_HOURS - 1)
    week_df = df.loc[week_start:week_end].copy()
    if len(week_df) != WEEK_LENGTH_HOURS:
        raise ValueError(f"No hay {WEEK_LENGTH_HOURS} horas. Obtenidas {len(week_df)}")
    return week_df

def get_input_window(df: pd.DataFrame, forecast_start: pd.Timestamp) -> pd.DataFrame:
    start = forecast_start - pd.Timedelta(hours=WINDOW_IN)
    end = forecast_start - pd.Timedelta(hours=1)
    window_df = df.loc[start:end].copy()
    if len(window_df) != WINDOW_IN:
        raise ValueError(f"Ventana de entrada incorrecta: {len(window_df)} horas")
    return window_df

def load_tree_model(model_kind: str, station: str):
    model_path = os.path.join(TREE_MODEL_BASE[model_kind], station, "model.pkl")
    with open(model_path, "rb") as f:
        model = pickle.load(f)
    return model

def load_rnn_bundle(model_kind: str, station: str):
    model_path = os.path.join(RNN_MODEL_BASE[model_kind], station, "model.keras")
    scaler_X_path = os.path.join(SCALERS_BASE, station, "scaler_X.pkl")
    scaler_y_path = os.path.join(SCALERS_BASE, station, "scaler_y.pkl")
    model = load_model(model_path)
    with open(scaler_X_path, "rb") as f:
        scaler_X = pickle.load(f)
    with open(scaler_y_path, "rb") as f:
        scaler_y = pickle.load(f)
    return model, scaler_X, scaler_y

def predict_tree_model(model, input_window: pd.DataFrame) -> np.ndarray:
    flat_input = input_window.values.reshape(1, -1)
    pred = model.predict(flat_input)
    if pred.ndim == 2:
        pred = pred.flatten()
    return pred[:WINDOW_OUT]

def predict_rnn_model(model, input_window: pd.DataFrame, scaler_X, scaler_y) -> tuple:
    X = scaler_X.transform(input_window.values)
    X = X.reshape(1, input_window.shape[0], input_window.shape[1])
    pred_norm = model.predict(X, verbose=0)
    if pred_norm.ndim == 2:
        pred_norm = pred_norm.flatten()
    pred_original = scaler_y.inverse_transform(pred_norm.reshape(-1, 1)).flatten()
    return pred_original[:WINDOW_OUT], pred_norm[:WINDOW_OUT]

# =============================================================================
# DIAGNÓSTICO
# =============================================================================
def diagnostic_week(station: str, week_start: pd.Timestamp, verbose=True):
    print(f"\n{'='*80}")
    print(f"DIAGNÓSTICO PARA {station} - Semana {week_start.strftime('%Y-%m-%d')}")
    print(f"{'='*80}")
    
    try:
        df = load_original_data(station)
    except Exception as e:
        print(f"ERROR cargando datos: {e}")
        return None
    
    if "O3" not in df.columns:
        print("ERROR: columna O3 no encontrada.")
        return None
    
    try:
        week_df = get_week_slice(df, week_start)
        forecast_start = week_start + pd.Timedelta(hours=WEEK_LENGTH_HOURS - FORECAST_HOURS)
        input_window = get_input_window(df, forecast_start)
    except Exception as e:
        print(f"ERROR extrayendo ventanas: {e}")
        return None
    
    y_true = week_df["O3"].values[-FORECAST_HOURS:]
    
    if verbose:
        print(f"\n--- ESTADÍSTICAS DE LA VENTANA DE ENTRADA (72h) ---")
        print(f"Shape: {input_window.shape}")
        print(f"Columnas: {list(input_window.columns)}")
        print(f"Rango de O3 en entrada: [{input_window['O3'].min():.1f}, {input_window['O3'].max():.1f}]")
        print(f"\n--- VALORES REALES A PREDECIR (primeros 10) ---")
        print(y_true[:10])
        print(f"Media real: {y_true.mean():.2f}, Desv: {y_true.std():.2f}")
    
    models = {}
    try:
        models["RF"] = load_tree_model("RF", station)
    except Exception as e:
        print(f"RF no disponible: {e}")
    try:
        models["XGBoost"] = load_tree_model("XGBoost", station)
    except Exception as e:
        print(f"XGBoost no disponible: {e}")
    try:
        gru_model, gru_sx, gru_sy = load_rnn_bundle("GRU", station)
        models["GRU"] = (gru_model, gru_sx, gru_sy)
    except Exception as e:
        print(f"GRU no disponible: {e}")
    try:
        lstm_model, lstm_sx, lstm_sy = load_rnn_bundle("LSTM", station)
        models["LSTM"] = (lstm_model, lstm_sx, lstm_sy)
    except Exception as e:
        print(f"LSTM no disponible: {e}")
    
    if len(models) != 4:
        print("No se pudieron cargar los cuatro modelos. Diagnóstico incompleto.")
        return None
    
    results = {}
    
    for name in ["RF", "XGBoost"]:
        pred = predict_tree_model(models[name], input_window)
        results[name] = pred
        if verbose:
            print(f"\n--- {name} ---")
            print(f"Predicción (primeros 10): {pred[:10]}")
            print(f"Media: {pred.mean():.2f}, Desv: {pred.std():.2f}, Min: {pred.min():.2f}, Max: {pred.max():.2f}")
            print(f"RMSE: {np.sqrt(mean_squared_error(y_true, pred)):.2f}")
            print(f"MAE: {mean_absolute_error(y_true, pred):.2f}")
    
    for name in ["GRU", "LSTM"]:
        model, sx, sy = models[name]
        pred_orig, pred_norm = predict_rnn_model(model, input_window, sx, sy)
        results[name] = pred_orig
        if verbose:
            print(f"\n--- {name} ---")
            print(f"Predicción original (primeros 10): {pred_orig[:10]}")
            print(f"Media original: {pred_orig.mean():.2f}, Desv: {pred_orig.std():.2f}, Min: {pred_orig.min():.2f}, Max: {pred_orig.max():.2f}")
            print(f"Predicción NORMALIZADA (primeros 10): {pred_norm[:10]}")
            print(f"Media normalizada: {pred_norm.mean():.2f}, Desv: {pred_norm.std():.2f}")
            print(f"RMSE: {np.sqrt(mean_squared_error(y_true, pred_orig)):.2f}")
            print(f"MAE: {mean_absolute_error(y_true, pred_orig):.2f}")
            # Información del scaler_y (compatible con StandardScaler y MinMaxScaler)
            if hasattr(sy, 'mean_'):
                print(f"Scaler_y tipo StandardScaler: mean={sy.mean_[0]:.2f}, scale={sy.scale_[0]:.2f}")
            elif hasattr(sy, 'min_'):
                print(f"Scaler_y tipo MinMaxScaler: min={sy.min_[0]:.2f}, scale={sy.scale_[0]:.2f} (rango: {sy.data_range_[0]:.2f})")
            else:
                print(f"Scaler_y tipo: {type(sy).__name__}")
            # Alertas
            if pred_norm.std() < 0.01:
                print("  *** ALERTA: La salida NORMALIZADA es casi constante. El modelo no está generando variabilidad. ***")
            if pred_orig.std() < 1.0:
                print("  *** ALERTA: La predicción DESNORMALIZADA es casi constante. Posible problema de escalado o modelo. ***")
    
    print(f"\n--- COMPARACIÓN DE MÉTRICAS EN ESTA SEMANA ---")
    for name in ["RF", "XGBoost", "GRU", "LSTM"]:
        rmse = np.sqrt(mean_squared_error(y_true, results[name]))
        mae = mean_absolute_error(y_true, results[name])
        print(f"{name:10s} | RMSE: {rmse:6.2f} | MAE: {mae:6.2f}")
    
    return {"y_true": y_true, "predictions": results, "week_start": week_start, "station": station}

# =============================================================================
# EJECUCIÓN
# =============================================================================
if __name__ == "__main__":
    week_start = pd.Timestamp(WEEK_START_STR)
    results = diagnostic_week(STATION, week_start, verbose=True)
    
    if results is not None:
        df_out = pd.DataFrame({
            "fecha": pd.date_range(start=results["week_start"] + pd.Timedelta(hours=WEEK_LENGTH_HOURS - FORECAST_HOURS),
                                   periods=FORECAST_HOURS, freq="h"),
            "real_O3": results["y_true"]
        })
        for name, pred in results["predictions"].items():
            df_out[f"pred_{name}"] = pred
        out_file = f"diagnostico_{results['station']}_{results['week_start'].strftime('%Y%m%d')}.csv"
        df_out.to_csv(out_file, index=False)
        print(f"\nArchivo CSV guardado: {out_file}")
    
    print("\nDiagnóstico completado.")


DIAGNÓSTICO PARA T1_E1_Alicante - Semana 2024-05-01

--- ESTADÍSTICAS DE LA VENTANA DE ENTRADA (72h) ---
Shape: (72, 23)
Columnas: ['NO', 'NO2', 'NOx', 'O3', 'Veloc.', 'Temp.', 'R.Sol.', 'Dist.', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'week_sin', 'week_cos', 'month_sin', 'month_cos', 'year', 'Direc.sin', 'Direc.cos', 'Angulosin', 'Angulocos', 'Estacion_1', 'Transecto_1']
Rango de O3 en entrada: [21.0, 119.0]

--- VALORES REALES A PREDECIR (primeros 10) ---
[35.66666667 30.         32.         31.66666667 29.33333333 28.33333333
 23.         32.33333333 53.33333333 60.33333333]
Media real: 69.53, Desv: 30.23

--- RF ---
Predicción (primeros 10): [63.76336489 61.51263281 59.83223723 58.28653122 54.85943106 48.1434276
 42.67262582 49.54906294 68.52662316 83.43226079]
Media: 80.86, Desv: 21.44, Min: 42.67, Max: 109.04
RMSE: 22.68
MAE: 18.15

--- XGBoost ---
Predicción (primeros 10): [60.975    53.815193 58.39526  57.68575  57.573616 53.355854 55.776054
 59.68956  73.07156  82.47312

In [11]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Diagnóstico semanal sin archivos de timestamps.
Reconstruye las fechas a partir de la fecha de inicio del conjunto.
"""

import os
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt

# =============================================================================
# CONFIGURACIÓN (MODIFICAR SEGÚN TUS RUTAS Y FECHAS)
# =============================================================================
BASE_DIR = os.path.expanduser("/Volumes/copia_seguridad1/enviar_benja/carpeta sin título/clean/Finales/")
STATION = "T1_E1_Alicante"

# Fechas de inicio de cada conjunto (formato YYYY-MM-DD HH:MM:SS)
START_DATES = {
    "test": pd.Timestamp("2025-01-03 00:00:00"),
    "val":  pd.Timestamp("2024-01-03 00:00:00")
}

# Umbral para detectar semanas problemáticas (RMSE_RNN > THRESHOLD * RMSE_XGB)
THRESHOLD_RATIO = 1.2

# Rutas
PARTITION_BASE = os.path.join(BASE_DIR, "windows_partitioned", "global")
ML_2D_DIR = os.path.join(PARTITION_BASE, "ml", STATION, "ml_2d")
DL_DIR = os.path.join(PARTITION_BASE, "dl", STATION)

RF_MODEL_PATH = os.path.join(BASE_DIR, "models", "random_forest", "global", STATION, "model.pkl")
XGB_MODEL_PATH = os.path.join(BASE_DIR, "models", "xgboost", "global", STATION, "model.pkl")
GRU_MODEL_PATH = os.path.join(BASE_DIR, "models", "rnn", "global", STATION, "model.keras")
LSTM_MODEL_PATH = os.path.join(BASE_DIR, "models", "lstm", "global", STATION, "model.keras")
SCALER_X_PATH = os.path.join(DL_DIR, "scaler_X.pkl")
SCALER_Y_PATH = os.path.join(DL_DIR, "scaler_y.pkl")

OUTPUT_DIR = os.path.join(BASE_DIR, "weekly_diagnostics")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================================================================
# FUNCIONES
# =============================================================================
def load_ml_data(set_name):
    X_path = os.path.join(ML_2D_DIR, f"{set_name}_X.npy")
    y_path = os.path.join(ML_2D_DIR, f"{set_name}_y.npy")
    if not (os.path.exists(X_path) and os.path.exists(y_path)):
        raise FileNotFoundError(f"Faltan archivos para {set_name} en {ML_2D_DIR}")
    X = np.load(X_path)
    y = np.load(y_path)
    return X, y

def load_dl_data(set_name):
    X_path = os.path.join(DL_DIR, f"{set_name}_X.npy")
    y_path = os.path.join(DL_DIR, f"{set_name}_y.npy")
    if not (os.path.exists(X_path) and os.path.exists(y_path)):
        raise FileNotFoundError(f"Faltan archivos para {set_name} en {DL_DIR}")
    X = np.load(X_path)
    y = np.load(y_path)
    if X.ndim == 2:
        n_features = X.shape[1] // 72
        X = X.reshape(-1, 72, n_features)
    return X, y

def load_models():
    models = {}
    if os.path.exists(RF_MODEL_PATH):
        with open(RF_MODEL_PATH, "rb") as f:
            models["RF"] = pickle.load(f)
        print("  RF cargado")
    if os.path.exists(XGB_MODEL_PATH):
        with open(XGB_MODEL_PATH, "rb") as f:
            models["XGBoost"] = pickle.load(f)
        print("  XGBoost cargado")
    if os.path.exists(GRU_MODEL_PATH) and os.path.exists(SCALER_X_PATH) and os.path.exists(SCALER_Y_PATH):
        with open(SCALER_X_PATH, "rb") as f:
            sx = pickle.load(f)
        with open(SCALER_Y_PATH, "rb") as f:
            sy = pickle.load(f)
        models["GRU"] = (load_model(GRU_MODEL_PATH), sx, sy)
        print("  GRU cargado")
    if os.path.exists(LSTM_MODEL_PATH) and os.path.exists(SCALER_X_PATH) and os.path.exists(SCALER_Y_PATH):
        with open(SCALER_X_PATH, "rb") as f:
            sx = pickle.load(f)
        with open(SCALER_Y_PATH, "rb") as f:
            sy = pickle.load(f)
        models["LSTM"] = (load_model(LSTM_MODEL_PATH), sx, sy)
        print("  LSTM cargado")
    return models

def predict_rnn(model, scaler_X, scaler_y, X_3d):
    orig_shape = X_3d.shape
    X_flat = X_3d.reshape(-1, orig_shape[-1])
    X_norm = scaler_X.transform(X_flat)
    X_norm = X_norm.reshape(orig_shape)
    pred_norm = model.predict(X_norm, verbose=0)
    if pred_norm.ndim == 2:
        pred_norm = pred_norm.reshape(pred_norm.shape[0], -1)
    pred_orig = scaler_y.inverse_transform(pred_norm.reshape(-1, 1)).reshape(pred_norm.shape)
    return pred_orig

def generate_timestamps(start_date, n_samples):
    """Genera una lista de timestamps para cada muestra, asumiendo paso de 1 hora."""
    return pd.date_range(start=start_date, periods=n_samples, freq='h')

def weekly_metrics_from_arrays(y_true, y_pred, timestamps):
    """Calcula métricas semanales (RMSE, MAE, R2) a partir de timestamps y arrays de predicciones."""
    weeks = []
    start_date = timestamps.min().normalize()
    end_date = timestamps.max().normalize() + pd.Timedelta(days=7)
    # Alinear al lunes
    days_to_monday = (start_date.weekday() - 0) % 7
    current = start_date + pd.Timedelta(days=days_to_monday)
    while current < end_date:
        week_end = current + pd.Timedelta(days=7)
        mask = (timestamps >= current) & (timestamps < week_end)
        if np.any(mask):
            true_flat = y_true[mask].flatten()
            pred_flat = y_pred[mask].flatten()
            rmse = np.sqrt(mean_squared_error(true_flat, pred_flat))
            mae = mean_absolute_error(true_flat, pred_flat)
            r2 = r2_score(true_flat, pred_flat)
            weeks.append({
                "week_start": current,
                "week_end": week_end,
                "RMSE": rmse,
                "MAE": mae,
                "R2": r2
            })
        current = week_end
    return pd.DataFrame(weeks)

def plot_weekly_rmse(df, station, set_name, model_names):
    if df.empty:
        return
    fig, ax = plt.subplots(figsize=(14, 6))
    for model in model_names:
        if f"{model}_RMSE" in df.columns:
            ax.plot(df["week_start"], df[f"{model}_RMSE"], marker='o', label=model)
    ax.set_title(f"RMSE semanal - {station} ({set_name})")
    ax.set_xlabel("Semana (inicio)")
    ax.set_ylabel("RMSE (µg/m³)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    fig.tight_layout()
    out_file = os.path.join(OUTPUT_DIR, f"{station}_{set_name}_weekly_rmse.png")
    fig.savefig(out_file, dpi=150)
    plt.close(fig)
    print(f"  Gráfica guardada: {out_file}")

def process_set(set_name, models, start_date):
    print(f"\nProcesando conjunto: {set_name}")
    try:
        X_ml, y_ml = load_ml_data(set_name)
        X_dl, y_dl = load_dl_data(set_name)
        assert len(y_ml) == len(y_dl), "Discrepancia en número de muestras entre ML y DL"
        n_samples = len(y_ml)
        timestamps = generate_timestamps(start_date, n_samples)
        y_true = y_ml
    except Exception as e:
        print(f"  Error cargando datos: {e}")
        return None

    predictions = {}

    # Árboles
    if "RF" in models:
        pred = models["RF"].predict(X_ml)
        if pred.ndim == 2 and pred.shape[1] != 72:
            pred = pred.reshape(-1, 72)
        predictions["RF"] = pred
    if "XGBoost" in models:
        pred = models["XGBoost"].predict(X_ml)
        if pred.ndim == 2 and pred.shape[1] != 72:
            pred = pred.reshape(-1, 72)
        predictions["XGBoost"] = pred

    # RNN
    if "GRU" in models:
        model, sx, sy = models["GRU"]
        pred = predict_rnn(model, sx, sy, X_dl)
        predictions["GRU"] = pred
    if "LSTM" in models:
        model, sx, sy = models["LSTM"]
        pred = predict_rnn(model, sx, sy, X_dl)
        predictions["LSTM"] = pred

    # Calcular métricas semanales para cada modelo
    weekly_dfs = {}
    for model_name, pred in predictions.items():
        df_week = weekly_metrics_from_arrays(y_true, pred, timestamps)
        df_week = df_week.rename(columns={
            "RMSE": f"{model_name}_RMSE",
            "MAE": f"{model_name}_MAE",
            "R2": f"{model_name}_R2"
        })
        # Guardar la columna week_end solo una vez (la tomamos del primer modelo)
        if not weekly_dfs:
            df_week = df_week.set_index("week_start")
        else:
            df_week = df_week.drop(columns=["week_end"]).set_index("week_start")
        weekly_dfs[model_name] = df_week

    # Combinar todos los DataFrames por índice (week_start)
    if not weekly_dfs:
        print("  No se generaron predicciones.")
        return None
    combined = pd.concat(weekly_dfs.values(), axis=1)
    combined = combined.reset_index().rename(columns={"index": "week_start"})

    # Guardar CSV
    csv_path = os.path.join(OUTPUT_DIR, f"{STATION}_{set_name}_weekly_metrics.csv")
    combined.to_csv(csv_path, index=False)
    print(f"  Guardado CSV: {csv_path}")

    # Identificar semanas problemáticas
    if "XGBoost_RMSE" in combined.columns and "GRU_RMSE" in combined.columns:
        combined["GRU_worse"] = combined["GRU_RMSE"] > (combined["XGBoost_RMSE"] * THRESHOLD_RATIO)
    if "XGBoost_RMSE" in combined.columns and "LSTM_RMSE" in combined.columns:
        combined["LSTM_worse"] = combined["LSTM_RMSE"] > (combined["XGBoost_RMSE"] * THRESHOLD_RATIO)
    problematic = combined[(combined.get("GRU_worse", False)) | (combined.get("LSTM_worse", False))]
    if not problematic.empty:
        print(f"  Semanas problemáticas encontradas: {len(problematic)}")
        prob_path = os.path.join(OUTPUT_DIR, f"{STATION}_{set_name}_problematic_weeks.csv")
        problematic.to_csv(prob_path, index=False)
        print(problematic[["week_start", "XGBoost_RMSE", "GRU_RMSE", "LSTM_RMSE"]].head(10))
    else:
        print("  No se encontraron semanas problemáticas.")

    plot_weekly_rmse(combined, STATION, set_name, list(predictions.keys()))
    return combined

def main():
    print(f"=== Diagnóstico semanal para {STATION} ===")
    models = load_models()
    if not models:
        print("No se pudo cargar ningún modelo. Abortando.")
        return

    if os.path.exists(os.path.join(ML_2D_DIR, "test_X.npy")):
        test_df = process_set("test", models, START_DATES["test"])
    else:
        print("No se encontraron datos de test.")

    if os.path.exists(os.path.join(ML_2D_DIR, "val_X.npy")):
        val_df = process_set("val", models, START_DATES["val"])
    else:
        print("No se encontraron datos de validación.")

    print("\nProceso completado.")

if __name__ == "__main__":
    main()

=== Diagnóstico semanal para T1_E1_Alicante ===
  RF cargado
  XGBoost cargado
  GRU cargado
  LSTM cargado

Procesando conjunto: test
  Guardado CSV: /Volumes/copia_seguridad1/enviar_benja/carpeta sin título/clean/Finales/weekly_diagnostics/T1_E1_Alicante_test_weekly_metrics.csv
  Semanas problemáticas encontradas: 50
  week_start  XGBoost_RMSE   GRU_RMSE  LSTM_RMSE
0 2025-01-07     23.302579  28.849428  28.380557
1 2025-01-14     22.720914  27.989573  44.923413
2 2025-01-21     25.888089  32.725129  40.562348
3 2025-01-28     18.458906  30.421076  33.949014
4 2025-02-04     14.947034  29.349827  36.813657
5 2025-02-11     19.805491  33.422319  38.251029
6 2025-02-18     24.891890  35.937984  30.631482
7 2025-02-25     26.879756  50.273726  24.400395
8 2025-03-04     18.062275  36.075232  25.613380
9 2025-03-11     18.697430  35.966259  25.638660
  Gráfica guardada: /Volumes/copia_seguridad1/enviar_benja/carpeta sin título/clean/Finales/weekly_diagnostics/T1_E1_Alicante_test_weekly_rm

In [14]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Diagnóstico semanal usando CSV original (corregido).
"""

import os
import pickle
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt

BASE_DIR = os.path.expanduser("/Volumes/copia_seguridad1/enviar_benja/carpeta sin título/clean/Finales/")
STATION = "T1_E1_Alicante"

# Fechas (ajusta según tu partición)
TEST_START = pd.Timestamp("2025-01-03 00:00:00")
VAL_START = pd.Timestamp("2024-01-03 00:00:00")
END_DATE = pd.Timestamp("2025-12-31 23:00:00")

WINDOW_IN = 72
WINDOW_OUT = 72
WEEK_LENGTH_HOURS = 168
FORECAST_HOURS = 72
THRESHOLD_RATIO = 1.2

ENCODED_DIR = os.path.join(BASE_DIR, "encoded", "ml", "global")
DATA_PATH = os.path.join(ENCODED_DIR, f"{STATION}.csv")

# Rutas corregidas
RF_MODEL_PATH = os.path.join(BASE_DIR, "models", "random_forest", "global", STATION, "model.pkl")
XGB_MODEL_PATH = os.path.join(BASE_DIR, "models", "xgboost", "global", STATION, "model.pkl")
GRU_MODEL_PATH = os.path.join(BASE_DIR, "models", "rnn", "global", STATION, "model.keras")
LSTM_MODEL_PATH = os.path.join(BASE_DIR, "models", "lstm", "global", STATION, "model.keras")
SCALERS_DIR = os.path.join(BASE_DIR, "windows_partitioned", "global", "dl", STATION)
SCALER_X_PATH = os.path.join(SCALERS_DIR, "scaler_X.pkl")
SCALER_Y_PATH = os.path.join(SCALERS_DIR, "scaler_y.pkl")

OUTPUT_DIR = os.path.join(BASE_DIR, "weekly_diagnostics1_csv")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ----------------------------------------------------------------------
def load_original_data(station: str) -> pd.DataFrame:
    csv_path = os.path.join(ENCODED_DIR, f"{station}.csv")
    df = pd.read_csv(csv_path)
    time_col = None
    for col in df.columns:
        if col.lower() in {"timestamp", "datetime", "fecha", "time"}:
            time_col = col
            break
    if time_col is None:
        time_col = df.columns[0]
    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
    df = df.dropna(subset=[time_col]).copy()
    df = df.set_index(time_col).sort_index()
    df = df.asfreq("h")
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.interpolate(method="time", limit_direction="both")
    df = df.bfill().ffill()
    return df

def get_week_slice(df: pd.DataFrame, week_start: pd.Timestamp) -> pd.DataFrame:
    week_end = week_start + pd.Timedelta(hours=WEEK_LENGTH_HOURS - 1)
    if week_end > df.index[-1]:
        raise ValueError("Semana fuera de rango")
    week_df = df.loc[week_start:week_end].copy()
    if len(week_df) != WEEK_LENGTH_HOURS:
        raise ValueError(f"No hay {WEEK_LENGTH_HOURS} horas (obtenidas {len(week_df)})")
    return week_df

def get_input_window(df: pd.DataFrame, forecast_start: pd.Timestamp) -> pd.DataFrame:
    start = forecast_start - pd.Timedelta(hours=WINDOW_IN)
    end = forecast_start - pd.Timedelta(hours=1)
    if start < df.index[0]:
        raise ValueError("Ventana fuera de rango")
    window_df = df.loc[start:end].copy()
    if len(window_df) != WINDOW_IN:
        raise ValueError(f"Ventana incorrecta: {len(window_df)}")
    return window_df

def predict_tree_model(model, input_window: pd.DataFrame) -> np.ndarray:
    flat_input = input_window.values.reshape(1, -1)
    pred = model.predict(flat_input)
    if pred.ndim == 2:
        pred = pred.flatten()
    return pred[:WINDOW_OUT]

def predict_rnn_model(model, input_window: pd.DataFrame, scaler_X, scaler_y) -> np.ndarray:
    X = scaler_X.transform(input_window.values)
    X = X.reshape(1, input_window.shape[0], input_window.shape[1])
    pred_norm = model.predict(X, verbose=0)
    if pred_norm.ndim == 2:
        pred_norm = pred_norm.flatten()
    pred_original = scaler_y.inverse_transform(pred_norm.reshape(-1, 1)).flatten()
    return pred_original[:WINDOW_OUT]

def generate_weeks(start_date, end_date, df_index):
    """Genera semanas completas que estén dentro del índice del DataFrame."""
    current = pd.Timestamp(start_date)
    days_to_monday = (current.weekday() - 0) % 7
    current = current + pd.Timedelta(days=days_to_monday)
    weeks = []
    while current <= df_index[-1] - pd.Timedelta(days=7):
        week_end = current + pd.Timedelta(days=7) - pd.Timedelta(hours=1)
        if week_end <= df_index[-1]:
            weeks.append((current, week_end + pd.Timedelta(hours=1)))
        current = current + pd.Timedelta(days=7)
    return weeks

def evaluate_week(df, week_start, models, scalers):
    try:
        week_df = get_week_slice(df, week_start)
    except Exception as e:
        return None
    forecast_start = week_start + pd.Timedelta(hours=WEEK_LENGTH_HOURS - FORECAST_HOURS)
    try:
        input_window = get_input_window(df, forecast_start)
    except Exception as e:
        return None
    y_true = week_df["O3"].values[-FORECAST_HOURS:]

    predictions = {}
    if "RF" in models:
        predictions["RF"] = predict_tree_model(models["RF"], input_window)
    if "XGBoost" in models:
        predictions["XGBoost"] = predict_tree_model(models["XGBoost"], input_window)
    if "GRU" in models and "GRU" in scalers:
        model, sx, sy = models["GRU"], scalers["GRU"][0], scalers["GRU"][1]
        predictions["GRU"] = predict_rnn_model(model, input_window, sx, sy)
    if "LSTM" in models and "LSTM" in scalers:
        model, sx, sy = models["LSTM"], scalers["LSTM"][0], scalers["LSTM"][1]
        predictions["LSTM"] = predict_rnn_model(model, input_window, sx, sy)

    metrics = {}
    for name, pred in predictions.items():
        metrics[name] = {
            "RMSE": np.sqrt(mean_squared_error(y_true, pred)),
            "MAE": mean_absolute_error(y_true, pred),
            "R2": r2_score(y_true, pred)
        }
    return metrics

def main():
    print(f"Diagnóstico para {STATION} usando CSV original")
    df = load_original_data(STATION)
    if "O3" not in df.columns:
        print("No hay O3")
        return

    models = {}
    scalers = {}
    if os.path.exists(RF_MODEL_PATH):
        with open(RF_MODEL_PATH, "rb") as f:
            models["RF"] = pickle.load(f)
        print("RF cargado")
    if os.path.exists(XGB_MODEL_PATH):
        with open(XGB_MODEL_PATH, "rb") as f:
            models["XGBoost"] = pickle.load(f)
        print("XGBoost cargado")
    if os.path.exists(GRU_MODEL_PATH) and os.path.exists(SCALER_X_PATH):
        with open(SCALER_X_PATH, "rb") as f:
            sx = pickle.load(f)
        with open(SCALER_Y_PATH, "rb") as f:
            sy = pickle.load(f)
        models["GRU"] = load_model(GRU_MODEL_PATH)
        scalers["GRU"] = (sx, sy)
        print("GRU cargado")
    if os.path.exists(LSTM_MODEL_PATH) and os.path.exists(SCALER_X_PATH):
        with open(SCALER_X_PATH, "rb") as f:
            sx = pickle.load(f)
        with open(SCALER_Y_PATH, "rb") as f:
            sy = pickle.load(f)
        models["LSTM"] = load_model(LSTM_MODEL_PATH)
        scalers["LSTM"] = (sx, sy)
        print("LSTM cargado")

    if not models:
        print("No hay modelos")
        return

    for set_name, start_date in [("test", TEST_START), ("val", VAL_START)]:
        print(f"\nProcesando {set_name}")
        weeks = generate_weeks(start_date, END_DATE, df.index)
        records = []
        for week_start, week_end in weeks:
            metrics = evaluate_week(df, week_start, models, scalers)
            if metrics is None:
                continue
            record = {"week_start": week_start, "week_end": week_end}
            for name, m in metrics.items():
                record[f"{name}_RMSE"] = m["RMSE"]
                record[f"{name}_MAE"] = m["MAE"]
                record[f"{name}_R2"] = m["R2"]
            records.append(record)
        if not records:
            continue
        df_weekly = pd.DataFrame(records)
        df_weekly.to_csv(os.path.join(OUTPUT_DIR, f"{STATION}_{set_name}_weekly_metrics.csv"), index=False)
        print(f"  Guardado CSV con {len(df_weekly)} semanas")
        # Gráfica
        fig, ax = plt.subplots(figsize=(14,6))
        for model in ["RF", "XGBoost", "GRU", "LSTM"]:
            if f"{model}_RMSE" in df_weekly.columns:
                ax.plot(df_weekly["week_start"], df_weekly[f"{model}_RMSE"], marker='o', label=model)
        ax.set_title(f"RMSE semanal - {STATION} ({set_name})")
        ax.set_xlabel("Semana (inicio)")
        ax.set_ylabel("RMSE (µg/m³)")
        ax.legend()
        ax.grid(True)
        plt.xticks(rotation=45)
        fig.tight_layout()
        fig.savefig(os.path.join(OUTPUT_DIR, f"{STATION}_{set_name}_weekly_rmse.png"), dpi=150)
        plt.close(fig)
        print(f"  Gráfica guardada")

if __name__ == "__main__":
    main()

Diagnóstico para T1_E1_Alicante usando CSV original
RF cargado
XGBoost cargado
GRU cargado
LSTM cargado

Procesando test
  Guardado CSV con 51 semanas
  Gráfica guardada

Procesando val
  Guardado CSV con 103 semanas
  Gráfica guardada


In [16]:
import os
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import load_model

# =============================================================================
# CONFIGURACIÓN
# =============================================================================
STATION = "T1_E1_Alicante"
WEEK_START = pd.Timestamp("2024-05-01 00:00:00")   # Semana a analizar

BASE_DIR = os.path.expanduser("/Volumes/copia_seguridad1/enviar_benja/carpeta sin título/clean/Finales/")
ENCODED_DIR = os.path.join(BASE_DIR, "encoded", "ml", "global")
OUTPUT_DIR = os.path.join(BASE_DIR, "comparative_forecasts_corrected")
os.makedirs(OUTPUT_DIR, exist_ok=True)

WINDOW_IN = 72
WINDOW_OUT = 72
WEEK_LENGTH_HOURS = 168
FORECAST_HOURS = 72

# Rutas de modelos (las correctas)
RF_PATH = os.path.join(BASE_DIR, "models", "random_forest", "global", STATION, "model.pkl")
XGB_PATH = os.path.join(BASE_DIR, "models", "xgboost", "global", STATION, "model.pkl")
GRU_PATH = os.path.join(BASE_DIR, "models", "rnn", "global", STATION, "model.keras")
LSTM_PATH = os.path.join(BASE_DIR, "models", "lstm", "global", STATION, "model.keras")
SCALER_X_PATH = os.path.join(BASE_DIR, "windows_partitioned", "global", "dl", STATION, "scaler_X.pkl")
SCALER_Y_PATH = os.path.join(BASE_DIR, "windows_partitioned", "global", "dl", STATION, "scaler_y.pkl")

MODEL_STYLE = {
    "RF": {"color": "tab:red", "linestyle": "--", "linewidth": 2.0, "label": "RF"},
    "XGBoost": {"color": "tab:green", "linestyle": "-.", "linewidth": 2.0, "label": "XGBoost"},
    "GRU": {"color": "tab:orange", "linestyle": ":", "linewidth": 2.5, "label": "GRU"},
    "LSTM": {"color": "tab:purple", "linestyle": (0, (5, 2)), "linewidth": 2.0, "label": "LSTM"},
}

# =============================================================================
# FUNCIONES
# =============================================================================
def load_original_data(station: str) -> pd.DataFrame:
    csv_path = os.path.join(ENCODED_DIR, f"{station}.csv")
    df = pd.read_csv(csv_path)
    time_col = None
    for col in df.columns:
        if col.lower() in {"timestamp", "datetime", "fecha", "time"}:
            time_col = col
            break
    if time_col is None:
        time_col = df.columns[0]
    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
    df = df.dropna(subset=[time_col]).copy()
    df = df.set_index(time_col).sort_index()
    df = df.asfreq("h")
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.interpolate(method="time", limit_direction="both")
    df = df.bfill().ffill()
    return df

def get_week_slice(df: pd.DataFrame, week_start: pd.Timestamp) -> pd.DataFrame:
    week_end = week_start + pd.Timedelta(hours=WEEK_LENGTH_HOURS - 1)
    week_df = df.loc[week_start:week_end].copy()
    if len(week_df) != WEEK_LENGTH_HOURS:
        raise ValueError(f"No hay {WEEK_LENGTH_HOURS} horas. Obtenidas {len(week_df)}")
    return week_df

def get_input_window(df: pd.DataFrame, forecast_start: pd.Timestamp) -> pd.DataFrame:
    start = forecast_start - pd.Timedelta(hours=WINDOW_IN)
    end = forecast_start - pd.Timedelta(hours=1)
    window_df = df.loc[start:end].copy()
    if len(window_df) != WINDOW_IN:
        raise ValueError(f"Ventana incorrecta: {len(window_df)}")
    return window_df

def predict_tree_model(model, input_window: pd.DataFrame) -> np.ndarray:
    flat_input = input_window.values.reshape(1, -1)
    pred = model.predict(flat_input)
    if pred.ndim == 2:
        pred = pred.flatten()
    return pred[:WINDOW_OUT]

def predict_rnn_model(model, input_window: pd.DataFrame, scaler_X, scaler_y) -> np.ndarray:
    X = scaler_X.transform(input_window.values)
    X = X.reshape(1, input_window.shape[0], input_window.shape[1])
    pred_norm = model.predict(X, verbose=0)
    if pred_norm.ndim == 2:
        pred_norm = pred_norm.flatten()
    pred_original = scaler_y.inverse_transform(pred_norm.reshape(-1, 1)).flatten()
    return pred_original[:WINDOW_OUT]

def plot_week(station, week_o3, predictions, week_start):
    forecast_start = week_start + pd.Timedelta(hours=WEEK_LENGTH_HOURS - FORECAST_HOURS)
    forecast_index = pd.date_range(start=forecast_start, periods=FORECAST_HOURS, freq="h")
    forecast_end = forecast_index[-1]

    fig, ax = plt.subplots(figsize=(15, 6))
    ax.plot(week_o3.index, week_o3.values, color="black", linewidth=1.8, label="O3 observado")
    ax.axvspan(forecast_start, forecast_end, alpha=0.08, color="gray", label="Periodo de predicción")
    for model_name in ["RF", "XGBoost", "GRU", "LSTM"]:
        pred = predictions[model_name]
        style = MODEL_STYLE[model_name]
        # pred es un array numpy, no tiene .values
        ax.plot(forecast_index, pred, color=style["color"], linestyle=style["linestyle"],
                linewidth=style["linewidth"], label=style["label"])
    ax.axvline(forecast_start, color="dimgray", linestyle=":", linewidth=1.5)
    ax.set_title(f"Serie semanal de O3 y predicciones de modelos\n{station}  |  Semana: {week_start.strftime('%Y-%m-%d')}")
    ax.set_xlabel("Fecha y hora")
    ax.set_ylabel("Concentración de O3 (µg/m³)")
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=3)
    fig.tight_layout()
    out_path = os.path.join(OUTPUT_DIR, f"{station}_week_{week_start.strftime('%Y%m%d')}_comparison.png")
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Gráfica guardada: {out_path}")

# =============================================================================
# MAIN
# =============================================================================
def main():
    print(f"Generando gráfica para {STATION}, semana {WEEK_START}")
    df = load_original_data(STATION)
    if "O3" not in df.columns:
        print("Error: columna O3 no encontrada")
        return

    try:
        week_df = get_week_slice(df, WEEK_START)
    except Exception as e:
        print(f"Error al extraer la semana: {e}")
        return

    forecast_start = WEEK_START + pd.Timedelta(hours=WEEK_LENGTH_HOURS - FORECAST_HOURS)
    try:
        input_window = get_input_window(df, forecast_start)
    except Exception as e:
        print(f"Error al extraer ventana: {e}")
        return

    # Cargar modelos
    models = {}
    scalers = {}
    if os.path.exists(RF_PATH):
        with open(RF_PATH, "rb") as f:
            models["RF"] = pickle.load(f)
        print("RF cargado")
    if os.path.exists(XGB_PATH):
        with open(XGB_PATH, "rb") as f:
            models["XGBoost"] = pickle.load(f)
        print("XGBoost cargado")
    if os.path.exists(GRU_PATH) and os.path.exists(SCALER_X_PATH):
        with open(SCALER_X_PATH, "rb") as f:
            sx = pickle.load(f)
        with open(SCALER_Y_PATH, "rb") as f:
            sy = pickle.load(f)
        models["GRU"] = load_model(GRU_PATH)
        scalers["GRU"] = (sx, sy)
        print("GRU cargado")
    if os.path.exists(LSTM_PATH) and os.path.exists(SCALER_X_PATH):
        with open(SCALER_X_PATH, "rb") as f:
            sx = pickle.load(f)
        with open(SCALER_Y_PATH, "rb") as f:
            sy = pickle.load(f)
        models["LSTM"] = load_model(LSTM_PATH)
        scalers["LSTM"] = (sx, sy)
        print("LSTM cargado")

    predictions = {}
    if "RF" in models:
        predictions["RF"] = predict_tree_model(models["RF"], input_window)
    if "XGBoost" in models:
        predictions["XGBoost"] = predict_tree_model(models["XGBoost"], input_window)
    if "GRU" in models and "GRU" in scalers:
        model, sx, sy = models["GRU"], scalers["GRU"][0], scalers["GRU"][1]
        predictions["GRU"] = predict_rnn_model(model, input_window, sx, sy)
    if "LSTM" in models and "LSTM" in scalers:
        model, sx, sy = models["LSTM"], scalers["LSTM"][0], scalers["LSTM"][1]
        predictions["LSTM"] = predict_rnn_model(model, input_window, sx, sy)

    if len(predictions) != 4:
        print("No están los cuatro modelos. Se omite gráfica.")
        return

    plot_week(STATION, week_df["O3"], predictions, WEEK_START)
    print("Proceso completado.")

if __name__ == "__main__":
    main()

Generando gráfica para T1_E1_Alicante, semana 2024-05-01 00:00:00
RF cargado
XGBoost cargado
GRU cargado
LSTM cargado
Gráfica guardada: /Volumes/copia_seguridad1/enviar_benja/carpeta sin título/clean/Finales/comparative_forecasts_corrected/T1_E1_Alicante_week_20240501_comparison.png
Proceso completado.
